In [2]:
# libraries

import open3d as o3d
import trimesh
import numpy as np
from pathlib import Path
import os
import random
from diffusion.preprocessing1 import *

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


ModuleNotFoundError: No module named 'diffusion'

In [ ]:
# parameters
sampling_type = 0
num_points = 2048
normalize = False
normalize_type = 1
make_watertight = False
keep_normals = False
fix_normals = False
object_number = 2

In [1]:
root = get_project_root()
print(f"Project Root identified as: {root}")

mesh_list = get_random_shapenet_meshes(seed=42, num_objects=10, root=root)
for path in mesh_list:
    print(path)


NameError: name 'get_project_root' is not defined

In [16]:
mesh = trimesh.load(filepath, force='mesh', skip_materials=True)

In [5]:
# Fix normals and winding order
mesh.fix_normals()

# Attempt to make it watertight (fills holes)
mesh.fill_holes()

# Remove zero-area triangles and duplicate vertices
mesh.process(validate=True)

# 1. Merge vertices that are at the exact same location
mesh.merge_vertices()


# 3. Remove "degenerate" faces (faces with zero area)
mesh.remove_infinite_values()
output_path = root / Path("data/preprocessed/preprocessing1") / f"mesh{output_number}.obj"
mesh.export(output_path)  # Save the cleaned mesh for inspection

print(f"Is watertight: {mesh.is_watertight}")

Is watertight: False


In [6]:
# Sample points from the surface
# returns a (point_num, 3) NumPy array

points = mesh.sample(num_points)

# Create a PointCloud object for Trimesh utilities
pc = trimesh.points.PointCloud(points)

In [7]:
def normalize_points(pcs):
    # Centering: Subtract the mean of the points
    centroid = np.mean(pcs, axis=0)
    pcs = pcs - centroid
    
    # Scaling: Find the furthest point and divide by that distance
    m = np.max(np.sqrt(np.sum(pcs**2, axis=1)))
    pcs = pcs / m
    return pcs

def normalize_statistically_points(pcs):
    # Centering: Subtract the mean of the points
    centroid = np.mean(pcs, axis=0)
    pcs = pcs - centroid
    
    # Scaling: Find the furthest point and divide by that distance
    v = np.var(pcs, axis=0)
    return pcs / np.sqrt(v)

if normalize:
    if normalize_type == 0:
        pc.vertices = normalize_points(pc.vertices)
    elif normalize_type == 1:
        pc.vertices = normalize_statistically_points(pc.vertices)

In [8]:

# Ensure output directory exists and save
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path = root / Path("data/preprocessed/preprocessing1") / f"pointcloud{output_number}.obj"
pc.export(output_path)
print(f"Saved preprocessed point cloud to: {output_path}")

Saved preprocessed point cloud to: /home/nikola/Projects/tum-adlr-ss26-07/data/preprocessed/preprocessing1/pointcloud1.obj
